In [1]:
# Este comando descarga el repositorio entero a una carpeta llamada 'TFMDS' en Colab.
#!git clone https://github.com/jmorala/TFMDS.git

# Inicializar directorios
Clonar repositorio github
Posicionarse en el directorio raíz

In [2]:
import os

# Detectar si estamos en Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Configurar el directorio de trabajo según el entorno
if IN_COLAB:
    os.chdir('TFMDS')
else:
    # Detectar si estamos en Codespaces o VS Code local
    if os.path.exists('/workspaces/TFMDS'):
        # Entorno Codespaces
        os.chdir('/workspaces/TFMDS')
    else:
        # En VS Code local, nos movemos al directorio raíz del proyecto
        # Usa raw string para evitar errores de escape en rutas Windows
        current_dir = r'C:\Users\jmora\Documents\TFMDS'
        os.chdir(current_dir)

# OPCIONAL: Para verificar que estás en la ruta correcta y ver las carpetas
print("Directorio de trabajo actual:", os.getcwd())

Directorio de trabajo actual: C:\Users\jmora\Documents\TFMDS


## Lectura de fichero y adaptación de los tipos


In [3]:
import pandas as pd

# Ruta relativa del archivo CSV
RUTA_DATOS_TRAIN = 'datos/df_train.csv'

# Cargar el archivo en un DataFrame de Pandas
dfSTventasTrain = pd.read_csv(RUTA_DATOS_TRAIN, sep=';',
    parse_dates=['idSecuencia'])

# Concatenar a dfSTVentas la lectura del fichero de Test
RUTA_DATOS_TEST = 'datos/df_test.csv'
dfSTventasTest = pd.read_csv(RUTA_DATOS_TEST, sep=';',
    parse_dates=['idSecuencia'])
dfSTventas = pd.concat([dfSTventasTrain, dfSTventasTest], ignore_index=True)

# Muestra las primeras filas y la información de las columnas para iniciar la exploración
print("Primeras filas del DataFrame:")
print(dfSTventas.head())

print("\nInformación de las columnas y tipos de datos:")
dfSTventas.info()

Primeras filas del DataFrame:
  idSecuencia  producto  udsVenta  bolPromocion  bolOpen  bolHoliday  \
0  2022-11-06         1         0             1        0           1   
1  2022-11-07         1        12             1        1           0   
2  2022-11-08         1        28             1        1           0   
3  2022-11-09         1        14             1        1           0   
4  2022-11-10         1        26             1        1           0   

   udsStock  rotura_stock  dia_semana  mes  ...  lag_ventas_4  lag_ventas_5  \
0       148         False           6   11  ...           0.0           0.0   
1       148         False           0   11  ...           0.0           0.0   
2       136         False           1   11  ...           0.0           0.0   
3       306         False           2   11  ...           0.0           0.0   
4       291         False           3   11  ...           0.0           0.0   

   lag_ventas_6  lag_ventas_7  media_mes_anterior  EWMA_corto 

# Generar dataframes para distintos tests

In [4]:
# Generar dataframe con la suma diaria de ventas: agrupar por día y sumar udsVenta
dfSTVentasTotalesTrain = dfSTventasTrain.groupby('idSecuencia')['udsVenta'].sum().reset_index()
dfSTVentasTotalesTest = dfSTventasTest.groupby('idSecuencia')['udsVenta'].sum().reset_index()
dfSTVentasTotales = dfSTventas.groupby('idSecuencia')['udsVenta'].sum().reset_index()

# Generar dataframe con la suma diaria de ventas por cluster
dfSTVentasClusterTrain = dfSTventasTrain.groupby(['idSecuencia', 'Cluster'])['udsVenta'].sum().reset_index()
dfSTVentasClusterTest = dfSTventasTest.groupby(['idSecuencia', 'Cluster'])['udsVenta'].sum().reset_index()
dfSTVentasCluster = dfSTventas.groupby(['idSecuencia', 'Cluster'])['udsVenta'].sum().reset_index()

# Naive

In [5]:
# ============================================================================
# MÉTODO NAIVE: Predicciones basadas en el último valor observado del train
# ============================================================================

from lib.metricas import calcular_metricas, resumen_metricas

metricas_naive = []  # acumulador de resultados

# ---------------------------------------------------------------
# 1. NAIVE SOBRE VENTAS DIARIAS TOTALES
# Predicción: todos los días del test = último valor del train
# ---------------------------------------------------------------
ultimo_train_total = dfSTVentasTotalesTrain['udsVenta'].iloc[-1]
naive_pred_total = pd.Series([ultimo_train_total] * len(dfSTVentasTotalesTest), index=dfSTVentasTotalesTest.index)

met_total = calcular_metricas(
    y=dfSTVentasTotalesTest['udsVenta'],
    y_pred=naive_pred_total,
    name='Naive_Total'
)
metricas_naive.append(met_total)
print(f"✅ Naive total -> último valor train: {ultimo_train_total}")

# ---------------------------------------------------------------
# 2. NAIVE POR CLUSTER (0..3)
# Para cada cluster: predicciones del test = último valor del train del cluster
# ---------------------------------------------------------------
clusters = sorted(dfSTVentasClusterTrain['Cluster'].dropna().unique())
for cl in clusters:
    df_tr_cl = dfSTVentasClusterTrain[dfSTVentasClusterTrain['Cluster'] == cl]
    df_te_cl = dfSTVentasClusterTest[dfSTVentasClusterTest['Cluster'] == cl]
    if df_tr_cl.empty or df_te_cl.empty:
        print(f"⚠️ Cluster {cl} sin datos suficientes (train o test vacío). Se omite.")
        continue
    ultimo_train_cluster = df_tr_cl['udsVenta'].iloc[-1]
    naive_pred_cluster = pd.Series([ultimo_train_cluster] * len(df_te_cl), index=df_te_cl.index)
    met_cl = calcular_metricas(
        y=df_te_cl['udsVenta'],
        y_pred=naive_pred_cluster,
        name=f'Naive_Cluster_{cl}'
    )
    metricas_naive.append(met_cl)
    print(f"✅ Naive cluster {cl} -> último valor train: {ultimo_train_cluster}")

# ---------------------------------------------------------------
# 3. NAIVE POR PRODUCTOS TOP (2 productos por cluster según ventas totales en train)
# Selección: sumar udsVenta en train por (Cluster, producto) y tomar top 2.
# Predicción: test = último valor diario del train para ese producto.
# ---------------------------------------------------------------
if 'producto' in dfSTventasTrain.columns:
    ventas_train_prod = (
        dfSTventasTrain.groupby(['Cluster', 'producto'], observed=True)['udsVenta']
        .sum()
        .reset_index()
    )
    for cl in clusters:
        top_prod_cl = (
            ventas_train_prod[ventas_train_prod['Cluster'] == cl]
            .sort_values('udsVenta', ascending=False)
            .head(2)
        )
        productos = top_prod_cl['producto'].tolist()
        if not productos:
            print(f"⚠️ Cluster {cl} sin productos para top 2.")
            continue
        for prod in productos:
            df_tr_prod = dfSTventasTrain[(dfSTventasTrain['Cluster'] == cl) & (dfSTventasTrain['producto'] == prod)]
            df_te_prod = dfSTventasTest[(dfSTventasTest['Cluster'] == cl) & (dfSTventasTest['producto'] == prod)]
            if df_tr_prod.empty or df_te_prod.empty:
                print(f"⚠️ Producto {prod} en cluster {cl} sin datos (train/test). Omitido.")
                continue
            ultimo_train_prod = df_tr_prod['udsVenta'].iloc[-1]
            naive_pred_prod = pd.Series([ultimo_train_prod] * len(df_te_prod), index=df_te_prod.index)
            met_prod = calcular_metricas(
                y=df_te_prod['udsVenta'],
                y_pred=naive_pred_prod,
                name=f'Naive_Prod_C{cl}_{prod}'
            )
            metricas_naive.append(met_prod)
            print(f"✅ Naive producto {prod} (Cluster {cl}) -> último valor train: {ultimo_train_prod}")
else:
    print("⚠️ No existe columna 'producto' en el dataset de train. Se omite sección de productos top.")

resumen_metricas(metricas_naive)

df_metricas_naive = pd.DataFrame(metricas_naive)

output_path_naive = 'datos/resultados_metricas_naive.csv'
df_metricas_naive.to_csv(output_path_naive, index=False)
print(f"\n💾 Métricas Naive guardadas en: {output_path_naive}")
print("="*100)

✅ Naive total -> último valor train: 1547
✅ Naive cluster 0 -> último valor train: 614
✅ Naive cluster 1 -> último valor train: 484
✅ Naive cluster 2 -> último valor train: 84
✅ Naive cluster 3 -> último valor train: 365
✅ Naive producto 413 (Cluster 0) -> último valor train: 2
✅ Naive producto 294 (Cluster 0) -> último valor train: 9
✅ Naive producto 144 (Cluster 1) -> último valor train: 5
✅ Naive producto 41 (Cluster 1) -> último valor train: 7
✅ Naive producto 1 (Cluster 2) -> último valor train: 7
✅ Naive producto 2 (Cluster 2) -> último valor train: 7
✅ Naive producto 257 (Cluster 3) -> último valor train: 0
✅ Naive producto 314 (Cluster 3) -> último valor train: 0

📊 RESUMEN DE MÉTRICAS
           Algoritmo      MAE         MSE     RMSE      R2  MAPE (%)  SMAPE (%)  RMSSE  MAE (%)
   Naive_Prod_C3_314   0.0000      0.0000   0.0000  1.0000       NaN       0.00    NaN      NaN
   Naive_Prod_C3_257   0.4194      1.3226   1.1500 -0.1534    100.00     200.00 0.8572   100.00
   Naive_

# Media de 7 días

In [6]:
# ============================================================================
# MÉTODO MEDIA 7 DÍAS: Predicción = media de los últimos 7 días del train
# ============================================================================

from lib.metricas import calcular_metricas, resumen_metricas
import pandas as pd
import numpy as np

WINDOW = 7
metricas_media7 = []

print("\n" + "="*100)
print(f"🧮 MEDIA MÓVIL BASE ({WINDOW} días) - Baseline")
print("="*100)

def media7_segmented(df_train, df_test, key_cols, name_prefix):
    """Genera predicciones constantes = media últimos WINDOW días del train.
    df_train y df_test deben contener 'idSecuencia','udsVenta' y opcionalmente columnas de key_cols.
    key_cols permite segmentar (Cluster, producto, etc.)."""

    resultados = []
    if key_cols:
        for clave_vals, grupo_train in df_train.groupby(key_cols, observed=True):
            if not isinstance(clave_vals, tuple):
                clave_vals = (clave_vals,)
            mask = np.ones(len(df_test), dtype=bool)
            for k, v in zip(key_cols, clave_vals):
                mask &= (df_test[k] == v)
            grupo_test = df_test[mask]

            if grupo_train.empty or grupo_test.empty:
                continue

            grupo_train_ord = grupo_train.sort_values('idSecuencia')
            ultimos = grupo_train_ord.tail(WINDOW)['udsVenta']
            media_val = ultimos.mean() if not ultimos.empty else grupo_train_ord['udsVenta'].mean()
            pred = pd.Series([media_val]*len(grupo_test), index=grupo_test.index)
            nombre = name_prefix + '_' + '_'.join(map(str, clave_vals))
            met = calcular_metricas(y=grupo_test['udsVenta'], y_pred=pred, name=nombre)
            resultados.append(met)
    else:
        df_train_ord = df_train.sort_values('idSecuencia')
        ultimos = df_train_ord.tail(WINDOW)['udsVenta']
        media_val = ultimos.mean() if not ultimos.empty else df_train_ord['udsVenta'].mean()
        pred = pd.Series([media_val]*len(df_test), index=df_test.index)
        met = calcular_metricas(y=df_test['udsVenta'], y_pred=pred, name=name_prefix)
        resultados.append(met)

    return resultados

# 1) MEDIA 7 DÍAS TOTAL
res_total = media7_segmented(
    df_train=dfSTVentasTotalesTrain[['idSecuencia','udsVenta']],
    df_test=dfSTVentasTotalesTest[['idSecuencia','udsVenta']],
    key_cols=[],
    name_prefix='Media7_Total'
)

metricas_media7.extend(res_total)

try:
    df_train_ord = dfSTVentasTotalesTrain.sort_values('idSecuencia')
    ultimos = df_train_ord.tail(WINDOW)['udsVenta']
    media_total = float(ultimos.mean() if not ultimos.empty else df_train_ord['udsVenta'].mean())
    print(f"✅ Media7 total -> media últimos {WINDOW} días del train: {media_total:.2f}")

except Exception as e:
    print(f"⚠️ No se pudo calcular detalle Media7 total: {e}")

# 2) MEDIA 7 DÍAS POR CLUSTER
res_cluster = media7_segmented(
    df_train=dfSTVentasClusterTrain[['idSecuencia','Cluster','udsVenta']],
    df_test=dfSTVentasClusterTest[['idSecuencia','Cluster','udsVenta']],
    key_cols=['Cluster'],
    name_prefix='Media7_Cluster'
)

metricas_media7.extend(res_cluster)
clusters_m7 = sorted(dfSTVentasClusterTrain['Cluster'].dropna().unique())

for cl in clusters_m7:
    df_tr_cl = dfSTVentasClusterTrain[dfSTVentasClusterTrain['Cluster'] == cl].sort_values('idSecuencia')
    if df_tr_cl.empty:
        continue

    ultimos = df_tr_cl.tail(WINDOW)['udsVenta']
    media_cl = float(ultimos.mean() if not ultimos.empty else df_tr_cl['udsVenta'].mean())
    print(f"✅ Media7 cluster {cl} -> media últimos {WINDOW} días: {media_cl:.2f}")

# 3) MEDIA 7 DÍAS TOP 2 PRODUCTOS POR CLUSTER
if 'producto' in dfSTventasTrain.columns:
    train_prod_daily = (
        dfSTventasTrain.groupby(['idSecuencia','Cluster','producto'], observed=True)['udsVenta']
        .sum().reset_index()
    )

    test_prod_daily = (
        dfSTventasTest.groupby(['idSecuencia','Cluster','producto'], observed=True)['udsVenta']
        .sum().reset_index()
    )

    total_train_prod = (
        train_prod_daily.groupby(['Cluster','producto'], observed=True)['udsVenta']
        .sum().reset_index()
    )

    for cl in sorted(total_train_prod['Cluster'].dropna().unique()):
        top2 = (
            total_train_prod[total_train_prod['Cluster'] == cl]
            .sort_values('udsVenta', ascending=False)
            .head(2)
        )

        productos = top2['producto'].tolist()
        if not productos:
            print(f"⚠️ Cluster {cl} sin productos top.")
            continue

        train_sub = train_prod_daily[(train_prod_daily['Cluster']==cl) & (train_prod_daily['producto'].isin(productos))]
        test_sub = test_prod_daily[(test_prod_daily['Cluster']==cl) & (test_prod_daily['producto'].isin(productos))]

        if train_sub.empty or test_sub.empty:
            print(f"⚠️ Cluster {cl} sin datos suficientes para productos top.")
            continue

        res_prod = media7_segmented(
            df_train=train_sub[['idSecuencia','Cluster','producto','udsVenta']],
            df_test=test_sub[['idSecuencia','Cluster','producto','udsVenta']],
            key_cols=['Cluster','producto'],
            name_prefix='Media7_Prod'
        )

        metricas_media7.extend(res_prod)

        for prod in productos:
            df_tr_prod = train_sub[(train_sub['producto'] == prod)].sort_values('idSecuencia')
            if df_tr_prod.empty:
                continue
            ultimos = df_tr_prod.tail(WINDOW)['udsVenta']
            media_prod = float(ultimos.mean() if not ultimos.empty else df_tr_prod['udsVenta'].mean())
            print(f"✅ Media7 producto {prod} (Cluster {cl}) -> media últimos {WINDOW} días: {media_prod:.2f}")

    print("✅ Media 7 días productos top por cluster calculada")
else:
    print("⚠️ Columna 'producto' no encontrada, se omite baseline por producto.")

# RESUMEN Y GUARDADO

resumen_metricas(metricas_media7)

df_metricas_media7 = pd.DataFrame(metricas_media7)

output_path_media7 = 'datos/resultados_metricas_media7.csv'
df_metricas_media7.to_csv(output_path_media7, index=False)
print(f"\n💾 Métricas Media 7 días guardadas en: {output_path_media7}")
print("="*100)



🧮 MEDIA MÓVIL BASE (7 días) - Baseline
✅ Media7 total -> media últimos 7 días del train: 1256.43
✅ Media7 cluster 0 -> media últimos 7 días: 496.14
✅ Media7 cluster 1 -> media últimos 7 días: 321.71
✅ Media7 cluster 2 -> media últimos 7 días: 76.71
✅ Media7 cluster 3 -> media últimos 7 días: 361.86
✅ Media7 producto 413 (Cluster 0) -> media últimos 7 días: 2.14
✅ Media7 producto 294 (Cluster 0) -> media últimos 7 días: 2.00
✅ Media7 producto 144 (Cluster 1) -> media últimos 7 días: 6.14
✅ Media7 producto 41 (Cluster 1) -> media últimos 7 días: 3.00
✅ Media7 producto 1 (Cluster 2) -> media últimos 7 días: 5.57
✅ Media7 producto 2 (Cluster 2) -> media últimos 7 días: 7.00
✅ Media7 producto 257 (Cluster 3) -> media últimos 7 días: 0.00
✅ Media7 producto 314 (Cluster 3) -> media últimos 7 días: 0.00
✅ Media 7 días productos top por cluster calculada

📊 RESUMEN DE MÉTRICAS
           Algoritmo      MAE         MSE     RMSE      R2  MAPE (%)  SMAPE (%)  RMSSE  MAE (%)
   Media7_Prod_3_314  